In [0]:
WITH cineclub_members AS (
    SELECT
        cde_id,
        CASE
            WHEN renewal_status LIKE 'Active%' THEN 'Active_Member'
            ELSE 'Past_Member'
        END AS member_segment
    FROM cpx_dataanalytics_gold.marketing.mkt_subscriber_profile_snapshot
    WHERE snapshot_date_id = DATE_FORMAT(DATE_SUB(CURRENT_DATE(), 1), 'yyyyMMdd')
),

scene_only_customers AS (
    SELECT DISTINCT
        CAST(t.CDE_ID AS STRING) AS cde_id,
        'Never_CineClub' AS member_segment
    FROM cpx_dataanalytics_gold.customer360.c360_f_transaction t
    LEFT ANTI JOIN cineclub_members m
        ON CAST(t.CDE_ID AS STRING) = m.cde_id
    WHERE t.SCENEFLAG = 'Yes'
      AND t.VisitDateId >= CAST(DATE_FORMAT(DATE_SUB(CURRENT_DATE(), 365), 'yyyyMMdd') AS INT)
      AND t.CDE_ID IS NOT NULL
),

all_customers AS (
    SELECT cde_id, member_segment FROM cineclub_members
    UNION ALL
    SELECT cde_id, member_segment FROM scene_only_customers
),

joined AS (
    SELECT
        m.cde_id,
        m.member_segment,
        r.segment AS rfm_segment
    FROM all_customers m
    INNER JOIN cpx_dataanalytics_gold.customer360.c360_f_rfm_model r
        ON m.cde_id = r.cde_id
    WHERE r.SnapshotDateId = DATE_FORMAT(DATE_SUB(CURRENT_DATE(), 1), 'yyyyMMdd')
)

SELECT
    member_segment,
    rfm_segment,
    COUNT(DISTINCT cde_id) AS cde_id_count
FROM joined
GROUP BY member_segment, rfm_segment
ORDER BY
    member_segment,
    CASE rfm_segment
        WHEN 'Champions'              THEN 1
        WHEN 'Potential Loyalist'     THEN 2
        WHEN 'Value Conscious Loyal'  THEN 3
        WHEN 'Promising'              THEN 4
        WHEN 'New'                    THEN 5
        WHEN 'At Risk'                THEN 6
        WHEN 'High Value Lapsed'      THEN 7
        WHEN 'Fleeting'               THEN 8
        ELSE 99
    END
;